# **Ollama Colab Runner**
# <img src='https://ollama.com/public/ollama.png' alt="Ollama"/>
When running this, ideally, select an instance with GPU:<br>
T4 for free ones, A100/L4 for paid subscribers<br><br>
Run each of the 3 cells, before running your prompt.<br>
If you interrupt execution, start the server again

In [1]:
# @title Install components
!curl https://ollama.ai/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  69415      0 --:--:-- --:--:-- --:--:-- 69534
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.downloa

In [2]:
# @title Start server
import subprocess
proccess = subprocess.Popen(['ollama', 'serve'])

In [3]:
# @title Select your model
model = "hoangquan456/qwen3-nothink:8b" # @param ["hoangquan456/qwen3-nothink:8b", "deepseek-r1:1.5b","deepseek-r1:7b","deepseek-r1:14b","deepseek-r1:32b","deepseek-r1:70b","deepseek-coder:1.3b","deepseek-coder:6.7b","deepseek-coder:33b","gemma3:12b","gemma3:27b","llama3.3:70b","mistral:7b","phi4:14b","qwen2.5:7b","qwen2.5:14b","qwen2.5:32b","qwen2.5-coder:7b","qwen2.5-coder:14b","qwen2.5-coder:32b"]
!ollama pull {model}

In [5]:
# @title Interacting with the model
question = "which is greater 0.9 or 0.11, do not reason just answer"
import ollama
response = ollama.chat(model, messages=[
  {
    'role': 'user',
    'content': question,
  },
])
print(response['message']['content'])

0.9 is greater.


In [ ]:
# @title Natural Gas Headline Filter - Complete Pipeline (Batch Processing)
# Configuration
INPUT_FILE = "gs://codeml/gas_headlines.json"
OUTPUT_FILE = "gas_headlines_filtered.json"
MAX_WORKERS = 5
BATCH_SIZE = 10

print("="*80)
print("NATURAL GAS HEADLINE FILTERING PIPELINE")
print("="*80)

# Install dependencies
print("\n[1/6] Installing dependencies...")
!pip install -q pandas tqdm

# Import libraries
print("[2/6] Importing libraries...")
import json
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import ollama
from google.colab import auth
auth.authenticate_user()

print("✓ Libraries imported successfully")

# Define batch relevance classification function
print("[3/6] Defining batch classification function...")

def check_headlines_batch_relevance(headlines_batch: list, model_name: str) -> list:
    """
    Check if headlines are relevant to natural gas industry.
    
    Args:
        headlines_batch: List of tuples (index, headline)
        model_name: The ollama model to use
        
    Returns:
        list: Indices of relevant headlines
    """
    system_prompt = """You are a news classification assistant specialized in the natural gas industry. 
Your task is to identify headlines with STRICT relevance to natural gas, natural gas prices, or the natural gas industry.

STRICT RELEVANCE CRITERIA:
- Must DIRECTLY mention natural gas, LNG (liquefied natural gas), or natural gas industry companies
- Must discuss current or future events/forecasts (NOT past/historical events)
- Slight implications or hints are NOT sufficient
- General energy topics without specific natural gas mention are NOT relevant
- Oil-only topics are NOT relevant unless they mention natural gas impact

Return only a JSON object with no markdown formatting."""
    
    # Build the headline list
    headlines_text = ""
    for idx, headline in headlines_batch:
        headlines_text += f"{idx}: {headline}\n"
    
    user_prompt = f"""Review these {len(headlines_batch)} headlines and return ONLY the indices of headlines that are STRICTLY relevant to natural gas.

Headlines:
{headlines_text}

STRICT Requirements:
1. Must DIRECTLY mention "natural gas", "LNG", or natural gas industry companies
2. Must discuss CURRENT or FUTURE events (no historical/past events)
3. Must be specifically about natural gas (not just general energy/oil)
4. Vague implications are NOT sufficient

Return ONLY a JSON object in this exact format with no other text:
{{"relevant_indices": [1, 3, 7]}}

If NO headlines are relevant, return:
{{"relevant_indices": []}}"""
    
    try:
        # Use ollama.chat
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        
        # Get the response content
        assistant_message = response['message']['content'].strip()
        
        # Clean up response (remove potential markdown formatting)
        if assistant_message.startswith('```json'):
            assistant_message = assistant_message[7:]
        if assistant_message.startswith('```'):
            assistant_message = assistant_message[3:]
        if assistant_message.endswith('```'):
            assistant_message = assistant_message[:-3]
        assistant_message = assistant_message.strip()
        
        # Parse JSON response
        result = json.loads(assistant_message)
        return result.get('relevant_indices', [])
        
    except Exception as e:
        print(f"Error processing batch: {e}")
        return []

def process_batch(batch_data):
    """Process a batch of rows and return results with relevance status."""
    # batch_data is a list of (index, row) tuples
    headlines_batch = [(idx, row['headline']) for idx, row in batch_data]
    relevant_indices = check_headlines_batch_relevance(headlines_batch, model)
    
    # Convert to set for O(1) lookup
    relevant_set = set(relevant_indices)
    
    results = []
    for idx, row in batch_data:
        results.append({
            'index': idx,
            'row': row.to_dict(),
            'is_relevant': idx in relevant_set
        })
    
    return results

print("✓ Batch classification function defined")

# Load data
print(f"[4/6] Loading data from {INPUT_FILE}...")
headlines_df = pd.read_json(INPUT_FILE)
print(f"✓ Loaded {len(headlines_df)} headlines")
print(f"  Columns: {headlines_df.columns.tolist()}")

# Create batches
print(f"\n[5/6] Creating batches and filtering headlines...")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Concurrent workers: {MAX_WORKERS}")
print(f"  Model: {model}")
print(f"{'='*80}")

batches = []
current_batch = []

for index, row in headlines_df.iterrows():
    current_batch.append((index, row))
    if len(current_batch) == BATCH_SIZE:
        batches.append(current_batch)
        current_batch = []

# Add remaining items as last batch
if current_batch:
    batches.append(current_batch)

print(f"✓ Created {len(batches)} batches")

filtered_headlines = []
relevance_stats = {'relevant': 0, 'not_relevant': 0}

# Use ThreadPoolExecutor for concurrent batch processing
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all batch tasks
    futures = {
        executor.submit(process_batch, batch): batch 
        for batch in batches
    }
    
    # Process completed futures with progress bar
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing batches"):
        try:
            results = future.result()
            
            for result in results:
                if result['is_relevant']:
                    filtered_headlines.append(result['row'])
                    relevance_stats['relevant'] += 1
                else:
                    relevance_stats['not_relevant'] += 1
                    
        except Exception as e:
            print(f"  Error processing batch: {str(e)[:100]}")
            # Count all items in failed batch as not relevant
            batch = futures[future]
            relevance_stats['not_relevant'] += len(batch)

# Create filtered DataFrame
filtered_df = pd.DataFrame(filtered_headlines)

# Display results
print(f"\n{'='*80}")
print("FILTERING RESULTS")
print(f"{'='*80}")
print(f"Total headlines processed: {len(headlines_df)}")
print(f"Relevant headlines: {relevance_stats['relevant']}")
print(f"Not relevant headlines: {relevance_stats['not_relevant']}")
print(f"Retention rate: {(relevance_stats['relevant'] / len(headlines_df) * 100):.2f}%")

# Save output
print(f"\n[6/6] Saving filtered data to {OUTPUT_FILE}...")

filtered_df_copy = filtered_df.copy()
for col in filtered_df_copy.select_dtypes(include=['datetime64']).columns:
    filtered_df_copy[col] = filtered_df_copy[col].dt.strftime('%Y-%m-%d')

filtered_data = filtered_df_copy.to_dict('records')

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

print(f"✓ Filtered headlines saved to: {OUTPUT_FILE}")
print(f"✓ Total entries saved: {len(filtered_data)}")

# Verify the output file
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    verification_data = json.load(f)
    print(f"✓ Successfully verified {len(verification_data)} entries in output file")

print(f"\n{'='*80}")
print("PIPELINE COMPLETE")
print(f"{'='*80}")
print(f"\nFiltered headlines preview:")
display(filtered_df.head(10))